# Day 26: SQL Subqueries & Window Functions

## Setup
This notebook covers two more advanced SQL tools — subqueries and window functions — both of which let you answer questions that a plain `SELECT`/`GROUP BY` can't handle on its own, like "which rows are above the overall average" or "rank rows within each group without collapsing them." A single `sales` table with a region and an amount per row is enough to demonstrate both, since the point here is the SQL technique, not the dataset itself. As in the previous SQL notebooks, `executescript()` creates the table and seeds it with a handful of rows in one call — this cell produces no output, it just sets the data up for the queries below.</cell id="cell-1">


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
# create the sales table and insert 5 rows in one script call
conn.executescript('''
    CREATE TABLE sales (id INTEGER PRIMARY KEY, region TEXT, amount INTEGER);
    INSERT INTO sales (region, amount) VALUES
        ('East', 100), ('East', 200), ('West', 150), ('West', 300), ('West', 50);
''')

## Subquery
A subquery is a complete `SELECT` statement nested inside another query, and it's evaluated first so its result can be used by the outer query — here, `(SELECT AVG(amount) FROM sales)` computes the overall average amount, and the outer query then filters for rows above that value. This pattern — "compare each row to some aggregate computed across the whole table" — comes up constantly: finding above-average performers, below-median prices, employees earning more than their department's average, etc. It's something you genuinely can't do with `WHERE` alone, since `WHERE` can't compute an aggregate over the whole table while also filtering individual rows in the same breath. The output shows only the rows where `amount` exceeds the average of all 5 amounts in the table.</cell id="cell-3">


In [ ]:
query = '''
    SELECT * FROM sales
    WHERE amount > (SELECT AVG(amount) FROM sales)
'''
# the inner (SELECT AVG...) subquery runs first, computing one number the outer WHERE then compares against
print(pd.read_sql_query(query, conn))

## RANK / ROW_NUMBER
`GROUP BY` collapses rows into one summary row per group, but sometimes you want to keep every individual row while still knowing its position *within* its group — like ranking each sale within its region without losing the row-level detail. Window functions do exactly this: `PARTITION BY region` splits the rows into per-region windows (similar to grouping), `ORDER BY amount DESC` decides the ranking order within each window, and `RANK()` assigns a rank number based on that order — all without merging rows together. This is extremely useful for tasks like "top N per category" or "find the best-selling product in each region." In the output, each region's rows are ranked separately starting from 1 (highest amount = rank 1), so East and West each have their own independent 1, 2, ... ranking.</cell id="cell-5">


In [ ]:
query = '''
    SELECT region, amount,
           RANK() OVER (PARTITION BY region ORDER BY amount DESC) AS rank_in_region
    FROM sales
'''
# PARTITION BY region -> rank restarts for each region; ORDER BY amount DESC -> highest amount gets rank 1
print(pd.read_sql_query(query, conn))

## Running Total with Window Function
`SUM() OVER` is another window function, but instead of ranking rows it accumulates a total across them — like the `cumsum()` we used in pandas, but expressed in SQL. Without `PARTITION BY`, the window covers the entire table, and `ORDER BY id` determines the order the running total accumulates in (by insertion order here). This is useful for things like a running account balance or year-to-date sales, directly from SQL, without needing to pull the data out into pandas first just to compute a cumulative sum. In the output, `running_total` starts at the first row's amount and keeps adding each subsequent row's amount, so the last row's `running_total` equals the sum of the entire `amount` column.</cell id="cell-7">


In [ ]:
query = '''
    SELECT region, amount,
           SUM(amount) OVER (ORDER BY id) AS running_total
    FROM sales
'''
# no PARTITION BY -> one running total across the whole table, accumulating in id order
print(pd.read_sql_query(query, conn))